In [1]:
import os
from pathlib import Path
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from supabase import create_client

In [2]:
print("Week 7 setup works")
print(pd.__version__)

Week 7 setup works
3.0.3


## Supabase connection test

In [3]:
dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd() / 'week7-python' / '.env',
    Path.cwd().parent / '.env',
    Path.cwd().parent / 'week7-python' / '.env',
    *[parent / '.env' for parent in Path.cwd().parents],
]
dotenv_path = next((path for path in dotenv_candidates if path.exists()), None)
load_dotenv(dotenv_path=dotenv_path)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

print(f"URL loaded: {bool(SUPABASE_URL)}")
print(f"KEY loaded: {bool(SUPABASE_KEY)}")

URL loaded: True
KEY loaded: True


In [4]:
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [5]:
response = supabase.table("sales").select("*").execute()

In [6]:
df = pd.DataFrame(response.data)
df.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,15235,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,15236,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,15237,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,15238,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,15239,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              1000 non-null   int64  
 1   sale_id         1000 non-null   int64  
 2   invoice_id      1000 non-null   str    
 3   sale_date       1000 non-null   str    
 4   customer_id     923 non-null    float64
 5   product_id      1000 non-null   int64  
 6   quantity        1000 non-null   int64  
 7   unit_price      1000 non-null   float64
 8   total_price     1000 non-null   float64
 9   channel         1000 non-null   str    
 10  store_location  688 non-null    str    
 11  payment_method  1000 non-null   str    
dtypes: float64(3), int64(4), str(5)
memory usage: 93.9 KB


## Load full sales table with pagination

In [8]:
batch_size = 1000
start = 0
all_rows = []

while True:
    response = (
        supabase.table("sales")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )

    rows = response.data

    if not rows:
        break

    all_rows.extend(rows)

    if len(rows) < batch_size:
        break

    start += batch_size

sales_df = pd.DataFrame(all_rows)
print(sales_df.shape)
sales_df.head()

(10118, 12)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,15235,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,15236,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,15237,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,15238,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,15239,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


Run the setup, Supabase connection, and full sales table cells before running this section.

## Basic sales analysis

In [9]:
sales_df["sale_date"] = pd.to_datetime(sales_df["sale_date"])

In [10]:
daily_sales = sales_df.groupby("sale_date")["total_price"].sum().reset_index()

In [11]:
daily_sales.head()

,sale_date,total_price
0,2023-01-01,3424.32
1,2023-01-02,3226.10
2,2023-01-03,1440.98
3,2023-01-04,2721.23
4,2023-01-05,3274.37


In [12]:
fig = px.line(
    daily_sales,
    x="sale_date",
    y="total_price",
    title="Müük ajas"
)
fig.show()

In [13]:
sales_df["year_month"] = sales_df["sale_date"].dt.to_period("M").astype(str)
monthly_sales = sales_df.groupby("year_month")["total_price"].sum().reset_index()

In [14]:
monthly_sales.head()

,year_month,total_price
0,2023-01,79735.03
1,2023-02,80345.68
2,2023-03,91499.55
3,2023-04,99914.07
4,2023-05,95316.25


In [15]:
fig = px.bar(
    monthly_sales,
    x="year_month",
    y="total_price",
    title="Kuupõhine müük"
)
fig.show()